# 用 Sciverse 查找论文全文证据

> 从检索片段出发，定位并读取原文完整段落作为可引用证据

---

**Sciverse Cookbook** | [文档首页](https://sciverse.space/docs#cookbook) | [申请 API Key](https://sciverse.space/docs#auth)

## 前置准备

1. 在 [Sciverse 控制台](https://sciverse.space/) 申请 API Token
2. 安装依赖：`pip install httpx anthropic`
3. 将下方 `sv-...` 替换为你的真实 Token


In [ ]:
# 安装依赖（Colab 环境）
!pip install -q httpx anthropic


## Step 1: 从检索结果获取定位信息

agentic-search 返回的每个 hit 包含 doc_id 和 offset


In [ ]:
# 假设已有检索结果
hit = {
    "doc_id": "af2_nature_2021",
    "chunk": "AlphaFold2 achieves atomic accuracy...",
    "offset": 12480,
    "score": 0.94
}

# 用 offset 定位原文位置
print(f"Will read from doc {hit['doc_id']} at offset {hit['offset']}")

## Step 2: 读取完整上下文

调用 content 接口，以 offset 为起点读取原文


In [ ]:
import httpx

async def get_fulltext(doc_id: str, offset: int = 0, limit: int = 4096):
    async with httpx.AsyncClient() as client:
        resp = await client.get(
            "https://api.sciverse.space/content",
            headers={"Authorization": "Bearer sv-..."},
            params={"doc_id": doc_id, "offset": offset, "limit": limit}
        )
        return resp.json()

# 读取 chunk 所在位置的完整上下文
result = await get_fulltext(hit["doc_id"], offset=max(0, hit["offset"] - 500), limit=4096)
print(result["content"][:200] + "...")
print(f"Has more: {result['more']}, next_offset: {result.get('next_offset')}")

## Step 3: 迭代读取（可选）

如果需要更多上下文，使用 next_offset 继续


In [ ]:
# 如果 more=True，可以继续读取
full_text = result["content"]
while result.get("more") and len(full_text) < 16000:
    result = await get_fulltext(
        hit["doc_id"],
        offset=result["next_offset"],
        limit=4096
    )
    full_text += result["content"]

print(f"Total context length: {len(full_text)} chars")

---

## 下一步

- 📖 [查看完整 API 文档](https://sciverse.space/docs#sciverse/api)
- 🔑 [申请 / 管理 API Token](https://sciverse.space/)
- 📚 [浏览更多 Cookbook 案例](https://sciverse.space/docs#cookbook)
- 💬 [加入开发者社群](https://sciverse.space/)
